In [ ]:
import requests

In [ ]:
url = "https://api.groq.com/openai/v1/chat/completions"

In [ ]:
import os
from dotenv import load_dotenv

# The key now lives in a local .env file (see .env.example) instead of
# being hardcoded here. The original key that used to be in this cell
# was exposed in plaintext and has been rotated/revoked.
load_dotenv()

headers = {
    "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
    "Content-Type": "application/json"
}

In [ ]:
def add(a, b):
    return a + b

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Adds two numbers together",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["a", "b"]
            }
        }
    }
]

In [ ]:
body={
    "model":"openai/gpt-oss-20b",
    "messages":[
            {"role":"system","content":"You are a polite assistant that will work for Netflix. Stay on topic with Netflix.You may use the calculator tool if the customer needs help with billing-related math (like calculating total costs).If theres any related question be free to answer. Ask the customer to contact the customer care whenever the issue is about billing or account specific issue and also if the problem dosent get solved after a few time. Try to end the conversation in polite mode. Also ask their suggestion of their fav genre of movies and which one is the best for them.Standard with ads: $8.99/month, 1080p, 2 simultaneous streams.Standard: $19.99/month, ad-free, 1080p, 2 simultaneous streams.Premium: $26.99/month, ad-free, 4K + HDR, 4 simultaneous streams. No free trial currently offered. Extra member add-on: $7.99/month (with ads) or $9.99/month (ad-free) — Standard/Premium only. Can cancel or pause anytime"},
            {"role":"user","content":"Hi.How can i cancel my subscription"},
{"role":"assistant","content":"Of course, you can cancel that by canceling the auto pay from your desired payment website. You'll be able to watch your fav movies till your subscription date ends even after cancelling."},
{"role":"user","content":"Can i suscribe again after cancelling?"},
{"role":"assistant","content":"Of Course!Don't worry about that. You can cancel and re susbscribe whenever you want."},
{"role":"user","content":"OK"},
{"role":"assistant","content":"Anything else I can help you with?"},
{"role":"user","content":"What plan is U101 on?"}
    ],
    "tools":tools
}


In [ ]:
response = requests.post(url,headers=headers, json=body)

In [ ]:
print(response.status_code)

200


In [ ]:
reply=response.json()
print(reply)

{'id': 'chatcmpl-e0446575-ba0d-4bc0-93e3-179630951e79', 'object': 'chat.completion', 'created': 1789915719, 'model': 'openai/gpt-oss-20b', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'reasoning': 'The user asks: "What plan is U101 on?" They refer to a user_id presumably "U101". According to instructions, we can use the tool functions to get the current plan for a user. The user wants plan for U101. We should call functions.get_user_plan with user_id "U101".\n\nWe should then respond with the plan information. Also note that the instructions say: "If there\'s any related question be free to answer." Also "Ask the customer to contact the customer care whenever the issue is about billing or account specific issue and also if the problem doesn\'t get solved after a few times." The request is a straightforward query, no billing or account issue? It\'s a question about current plan. It\'s an account specific question. We should probably ask them to contact customer care if it\'

In [ ]:
tool_call = response.json()["choices"][0]["message"]["tool_calls"][0]
function_name = tool_call["function"]["name"]
arguments = tool_call["function"]["arguments"]
print(function_name)
print(arguments)

add
{"a":26.99,"b":7.99}


In [ ]:
import json
parsed_arguments = json.loads(arguments)
print(parsed_arguments)
print(type(parsed_arguments))


{'a': 26.99, 'b': 7.99}
<class 'dict'>


In [ ]:
result = add(parsed_arguments["a"], parsed_arguments["b"])
print(result)

34.98


In [ ]:
body["messages"].append(response.json()["choices"][0]["message"])

body["messages"].append({
    "role": "tool",
    "tool_call_id": tool_call["id"],
    "content": str(result)
})

In [ ]:
response2 = requests.post(url, headers=headers, json=body)
print(response2.json()["choices"][0]["message"]["content"])

The total monthly cost would be **$34.98** – that’s $26.99 for Premium plus $7.99 for the extra member add‑on with ads.

If you have any other questions—whether it’s about plans, device setup, or recommendations—I’m happy to help. Also, feel free to let me know your favorite genre or the best movie you’ve seen on Netflix—I can suggest similar titles you might enjoy!


In [ ]:
users_db = {
    "U101": {"name": "Aditi Sharma", "email": "aditi.sharma@gmail.com", "plan": "Premium"},
    "U102": {"name": "Rohan Mehta", "email": "rohan.mehta@gmail.com", "plan": "Standard with ads"},
    "U103": {"name": "Priya Nair", "email": "priya.nair@gmail.com", "plan": "Standard"},
}

In [ ]:
def get_user_plan(user_id):
  return users_db[user_id]['plan']

In [ ]:
print(get_user_plan("U101"))

Premium


In [ ]:
tools.append({
    "type": "function",
    "function": {
        "name": "get_user_plan",
        "description": "Gets the current plan for a user",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string"}
            },
            "required": ["user_id"]
        }
    }
})

In [ ]:
print(tools)

[{'type': 'function', 'function': {'name': 'add', 'description': 'Adds two numbers together', 'parameters': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}}}, {'type': 'function', 'function': {'name': 'get_user_plan', 'description': 'Gets the current plan for a user', 'parameters': {'type': 'object', 'properties': {'user_id': {'type': 'string'}}, 'required': ['user_id']}}}]


In [ ]:
tool_call = response.json()["choices"][0]["message"]["tool_calls"][0]
function_name = tool_call["function"]["name"]
arguments = json.loads(tool_call["function"]["arguments"])

In [ ]:
print(arguments)

{'user_id': 'U101'}


In [ ]:
result = get_user_plan(arguments["user_id"])
print(result)

Premium


In [ ]:
body["messages"].append(response.json()["choices"][0]["message"])
body["messages"].append({
    "role": "tool",
    "tool_call_id": tool_call["id"],
    "content": str(result)
})

response2 = requests.post(url, headers=headers, json=body)
reply = response2.json()["choices"][0]["message"]["content"]
print(reply)

User U101 is currently on the **Premium** plan – ad‑free, 4K + HDR, and allows up to 4 simultaneous streams.  

If you ever want to switch plans, pause, or have any billing questions, feel free to reach out to our Customer Care team for personalized support.  

While I’m here, what’s your favorite movie genre? And if you had to pick just one movie that you’d love to watch on Netflix, what would it be?
